In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_dir = "../save_improved"

model = AutoModelForCausalLM.from_pretrained(
    model_dir,
    torch_dtype=torch.float16,
    device_map="cpu"
)

tokenizer = AutoTokenizer.from_pretrained(model_dir)

model.eval()

c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 4824.64it/s]
LlamaForCausalLM LOAD REPORT from: ../save_improved
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
model.layers.{0...31}.qkt_smooth_scale                         | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.o_proj.weight_quantizer.zeros  | UNEXPECTED |  | 
model.layers.{0...31}.mlp.up_proj.weight_quantizer.scales      | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.k_proj.weight_quantizer.scales | UNEXPECTED |  | 
model.layers.{0...31}.self_attn.o_proj.weight_quantizer.scales | UN

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (up_proj): Linear(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((4096,), eps=1e-05)
   

In [2]:
text = "Quantization test"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

In [ ]:
text = "Quantization test"

inputs = tokenizer(
    text,
    return_tensors="pt"
)

input_ids = inputs["input_ids"]
attention_mask = inputs["attention_mask"]

In [3]:
torch.onnx.export(
    model,
    (input_ids, attention_mask),
    "../save_improved/llama_quantized.onnx",
    input_names=["input_ids", "attention_mask"],
    output_names=["logits"],
    opset_version=18,
    dynamic_axes={
        "input_ids": {0: "batch", 1: "sequence"},
        "attention_mask": {0: "batch", 1: "sequence"},
        "logits": {0: "batch", 1: "sequence"}
    }
)

C:\Users\CT-PROJECT\AppData\Local\Temp\ipykernel_8784\2967528482.py:1: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0312 16:32:17.611000 8784 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'input' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 16:32:17.613000 8784 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for parameter 'boxes' from (input, boxes, output_size: 'Sequence[int]', spatial_scale: 'float' = 1.0, sampling_ratio: 'int' = -1, aligned: 'bool' = False). Treating as an Input.
W0312 16:32:17.615000 8784 Lib\site-packages\torch\onnx\_internal\exporter\_schemas.py:455] Missing annotation for paramet

[torch.onnx] Obtain model graph for `LlamaForCausalLM([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `LlamaForCausalLM([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...


C:\Users\CT-PROJECT\AppData\Local\Programs\Python\Python314\Lib\copyreg.py:104: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


c:\Users\CT-PROJECT\Documents\Team10_FYP\.venv\Lib\site-packages\torch\onnx\_internal\exporter\_onnx_program.py:460: UserWarning: # The axis name: sequence will not be used, since it shares the same shape constraints with another axis: sequence.
  rename_mapping = _dynamic_shapes.create_rename_mapping(


Applied 150 of general pattern rewrite rules.


ONNXProgram(
    model=
        <
            ir_version=10,
            opset_imports={'': 18},
            producer_name='pytorch',
            producer_version='2.10.0+cu126',
            domain=None,
            model_version=None,
        >
        graph(
            name=main_graph,
            inputs=(
                %"input_ids"<INT64,[batch,sequence]>,
                %"attention_mask"<INT64,[batch,sequence]>
            ),
            outputs=(
                %"logits"<FLOAT16,[1,sequence,32000]>
            ),
            initializers=(
                %"model.embed_tokens.weight"<FLOAT16,[32000,4096]>{TorchTensor(...)},
                %"model.layers.0.input_layernorm.weight"<FLOAT16,[4096]>{TorchTensor(...)},
                %"model.layers.0.post_attention_layernorm.weight"<FLOAT16,[4096]>{TorchTensor(...)},
                %"model.layers.1.input_layernorm.weight"<FLOAT16,[4096]>{TorchTensor(...)},
                %"model.layers.1.post_attention_layernorm.weight"<FLOAT16

In [7]:
from onnxruntime.quantization import quantize_dynamic, QuantType

quantize_dynamic(
    model_input="../save_improved/llama_quantized.onnx",
    model_output="../save_improved/llama_final_int8.onnx",
    weight_type=QuantType.QInt8,
    # This ensures that the MatMul operations in Attention 
    # are handled correctly without overflowing
    per_channel=True, 
    reduce_range=True, # Recommended for many hardware backends
    # These ops are usually the best candidates for dynamic quantization
    use_external_data_format=True, 
    # Optional: keeps the graph cleaner for large models
    extra_options={"DisableShapeInference": True},
    nodes_to_quantize=None 
)